# A2 smoke run — `VLMVerifier` trên 3 cửa sổ thật (Phước)

Mục đích: chạy **đường thật** (Qwen2.5-VL-3B, 4-bit nf4) end-to-end trên 3 window thật để chứng minh
`VLMVerifier.verify` trả `VerifyResult` thật — đây là acceptance criterion của **task A2**.

**KHÔNG phải benchmark**: không đánh giá accuracy (việc của A5), không đo tốc độ chuẩn (A1 đã đo trên T4:
4.45 s/window). Chỉ kiểm tra đường ống: nạp model → chat template → generate → parse JSON.

**Bắt buộc:** Runtime → Change runtime type → **T4 GPU**.

3 ca chọn sẵn (2 anomaly + 1 normal, mỗi window 4 frame 320x240):

| window | label | gt_event | video |
|---|---|---|---|
| `w01` | anomaly | Arrest | `Arrest024_x264` |
| `w02` | anomaly | Shoplifting | `Shoplifting001_x264` |
| `w09` | normal | Normal | `Normal_Videos585_x264` |

In [ ]:
# Cell 1 — xác nhận GPU. Không phải T4 -> Runtime > Change runtime type > T4 GPU.
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Cần GPU: Runtime > Change runtime type > T4 GPU"

In [ ]:
# Cell 2 — code + dữ liệu: clone đúng nhánh A2, giải nén 16 cửa sổ đã commit (notes/bench_windows.zip).
import glob
import os

REPO = "/content/vlm-anomalies-detection"
BRANCH = "feat/phuoc-a2-verifier"
if not os.path.isdir(REPO):
    !git clone -b {BRANCH} https://github.com/Kyosaki6/vlm-anomalies-detection.git {REPO}
os.chdir(REPO)
!git pull origin {BRANCH}
!unzip -oq notes/bench_windows.zip -d .
print("frames:", len(glob.glob("bench_windows/w*/frame_*.jpg")))

In [ ]:
# Cell 3 — deps VLM. GIỮ torch của Colab (cài lại torch làm hỏng CUDA wheels).
!pip install -q -r requirements-vlm.txt
!pip freeze | grep -iE "^(torch|transformers|bitsandbytes|accelerate)=="

In [ ]:
# Cell 4 — tải weight TRƯỚC (không tính thời gian tải vào smoke). ~6.5 GB.
!huggingface-cli download Qwen/Qwen2.5-VL-3B-Instruct > /dev/null && echo "weights OK"

## Chạy smoke — 3 cửa sổ

`VLMVerifier()` mặc định đúng handoff A1: `Qwen/Qwen2.5-VL-3B-Instruct`, 4-bit nf4, prompt `verifier`
từ `configs/prompts.yaml` (hash `3d71fdba`), greedy, `max_new_tokens=128`.

Có 1 lần **warmup** trên `w09` (không ghi vào kết quả) để thời gian window đầu không gồm load model.

In [ ]:
# Cell 5 — smoke run: warmup + 3 cửa sổ (2 anomaly + 1 normal), đo thời gian + VRAM.
import json
import time

import torch
from PIL import Image

from vad.verify.vlm import DEFAULT_MODEL_ID, VLMVerifier

PICK = ["w01", "w02", "w09"]
data = json.load(open("notes/vlm-bench-windows.json", encoding="utf-8-sig"))
windows = {w["window_id"]: w for w in data["windows"]}

v = VLMVerifier()  # default: Qwen2.5-VL-3B, 4-bit nf4, prompt verifier
print("model:", DEFAULT_MODEL_ID)


def frames_of(wid):
    w = windows[wid]
    return w, [Image.open(p).convert("RGB") for p in w["frame_paths"]]


def context_of(w):
    return {"video_id": w["video_id"], "start_s": w["segment"][0] / w["fps"],
            "end_s": w["segment"][1] / w["fps"]}


# Warmup: lần đầu gồm load model 4-bit + CUDA context -> không tính vào kết quả.
w, frames = frames_of("w09")
t0 = time.perf_counter()
v.verify(frames, context_of(w))
print(f"warmup (load + 1 call): {time.perf_counter() - t0:.1f}s")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

rows = []
for wid in PICK:
    w, frames = frames_of(wid)
    t0 = time.perf_counter()
    r = v.verify(frames, context_of(w))
    dt = time.perf_counter() - t0
    rows.append({"window_id": wid, "label": w["label"], "gt_event": w["gt_event"],
                 "is_anomaly": r.is_anomaly, "event_type": r.event_type.value,
                 "caption": r.caption, "confidence": round(r.confidence, 3),
                 "seconds": round(dt, 2)})
    print(f"{wid} [{w['label']}/{w['gt_event']}] -> is_anomaly={r.is_anomaly} "
          f"type={r.event_type.value} conf={r.confidence:.2f} {dt:.1f}s")
    print(f"   caption: {r.caption}")

peak_alloc = torch.cuda.max_memory_allocated() / 1024**3
peak_reserved = torch.cuda.max_memory_reserved() / 1024**3
print(f"\nVRAM peak: allocated {peak_alloc:.2f} GB | reserved {peak_reserved:.2f} GB")
print("smoke:", "PASS (3 VerifyResult thật)" if len(rows) == 3 else "FAIL")

In [ ]:
# Cell 6 — lưu kết quả (sống sót qua disconnect). Bỏ qua nếu không muốn mount Drive.
import json

try:
    from google.colab import drive

    drive.mount("/content/drive")
    out = "/content/drive/MyDrive/vlm_a1/a2_smoke.json"
    json.dump({"model": DEFAULT_MODEL_ID, "windows": rows,
               "peak_allocated_gb": round(peak_alloc, 2),
               "peak_reserved_gb": round(peak_reserved, 2)},
              open(out, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print("saved:", out)
except Exception as e:  # không mount Drive cũng không sao, kết quả đã in ở Cell 5
    print("không lưu Drive:", e)

## Đọc kết quả thế nào

- **Kỳ vọng** (không phải cam kết): `w01`/`w02` -> `is_anomaly=True`, `w09` -> `False`. n=3 **không phải
  accuracy** — lệch 1 ca thì ghi nhận, không kết luận gì (A5 mới là số liệu).
- `confidence` của 3B gần như **hằng số 0.95** -> không dùng làm tín hiệu.
- `event_type` có bias "Robbery" -> chỉ tin **caption**; typing là việc của C3 (typer riêng).
- Parse hỏng 2 lần -> `RuntimeError`: **đó là bug thật**, dán raw output vào PR và báo lại, đừng sửa prompt
  tại chỗ (đổi prompt = phải re-test 16 cửa sổ, xem `notes/vlm-selection.md` §7).

## Ghi vào PR #16

Dán bảng 3 dòng (`window_id`, `is_anomaly`, `event_type`, `caption`, `confidence`, `seconds`) + VRAM peak +
phiên bản `transformers`/`bitsandbytes` ở Cell 3. Đủ để tick acceptance: *"3-clip smoke run returns a real
`VerifyResult`"*.